In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_january_data():
    # API Endpoints
    actuals_url = "https://api.bmrs.elexon.co.uk/v1/datasets/FUELHH/stream"
    forecast_url = "https://api.bmrs.elexon.co.uk/v1/datasets/WINDFOR/stream"

    # 1. Fetch Actuals (FUELHH)
    # Filter for 'WIND' fuelType
    res_actuals = requests.get(actuals_url).json()
    actuals = [
        {"startTime": i["startTime"], "actualGen": i["generation"]}
        for i in res_actuals if i.get("fuelType") == "WIND"
    ]
    df_actuals = pd.DataFrame(actuals)

    # 2. Fetch Forecasts (WINDFOR)
    # Rules: Jan 2024, Horizon 0-48h
    res_forecasts = requests.get(forecast_url).json()
    forecasts = [
        {"startTime": i["startTime"], "publishTime": i["publishTime"], "forecastGen": i["generation"]}
        for i in res_forecasts
    ]
    df_forecasts = pd.DataFrame(forecasts)

    # 3. Merge and Save
    df_merged = pd.merge(df_actuals, df_forecasts, on="startTime", how="inner")
    df_merged.to_csv('jan_wind_data.csv', index=False)
    print("Dataset saved as jan_wind_data.csv")

if __name__ == "__main__":
    fetch_january_data()